# 超参数调优

RL 训练对超参数非常敏感。本节通过真实实验数据，展示三个关键超参数对训练的影响：entropy_coeff、kl_loss_coef 和学习率。

---

## 1. entropy_coeff

entropy_coeff 控制 Entropy Bonus 的强度，直接影响模型的探索能力。

### 实验数据

以下使用同一个 SFT 模型（Qwen3-1.7B-Wordle-SFT），仅改变 entropy_coeff：

| entropy_coeff | step 5 entropy | step 25 entropy | 结果 |
|---------------|---------------|-----------------|------|
| 0（默认） | 0.54 | 0.45 | entropy 持续下降，step 100 后崩塌到 0 |
| 0.001 | 0.56 | 0.48 | 下降稍缓，但仍可能崩塌 |
| 0.005 | 0.58 | 0.52 | 缓慢下降，需观察长期趋势 |
| 0.01 | 0.61 | **4.51** | entropy 失控飙升 |

### 分析

```text
entropy_coeff 的影响

entropy_coeff=0 (无 bonus)
  pg_loss 完全主导 -> entropy 持续下降 -> 最终崩塌

entropy_coeff=0.01 (过强)
  entropy_bonus 压过 pg_loss -> entropy 飙升 -> 模型过度探索
  25步内 entropy 从 0.56 飙到 4.51，指数增长
  reward 上升但 correct 不涨 (靠 partial 而非猜中)

entropy_coeff=0.005 (适中)
  三种力平衡 -> entropy 缓慢下降 -> 训练稳定
```

---

## 2. kl_loss_coef

kl_loss_coef 控制 KL 正则化强度，限制当前策略与参考策略之间的距离。

| kl_loss_coef | 效果 | 风险 |
|-------------|------|------|
| 0.001 | 轻微约束 | 可能不足以阻止漂移 |
| 0.01 | 中等约束 | 配合 entropy_coeff 使用 |
| 0.1 | 强约束 | 模型无法学习新策略 |

### KL loss 与 entropy 的关系

KL loss 和 entropy bonus 是互补的稳定性机制：

```text
KL loss: 拉回参考策略 (限制漂移)
  kl_loss = KL(pi || pi_ref)
  偏离越远 -> kl_loss 越大 -> 梯度拉回

entropy_bonus: 推高探索 (防止崩塌)
  entropy_bonus = -entropy_coeff * H(pi)
  熵越低 -> 惩罚越大 -> 梯度推高

两者协同: KL 限制方向, entropy 限制分布形状
```

---

## 3. 学习率（Actor LR）

学习率决定每次更新的步长，直接影响训练的稳定性和收敛速度。

| ACTOR_LR | 效果 | 风险 |
|----------|------|------|
| 1e-7 | 更新太慢 | 训练效率低 |
| 1e-6 | 缓慢稳定 | 当前配置，推荐 |
| 1e-5 | 更新较快 | 可能导致梯度爆炸 |
| 1e-4 | 更新过快 | 训练不稳定 |

> RL 的学习率通常比 SFT 低 1-2 个数量级，因为 RL 更新基于 noisy 的优势估计，需要更保守的步长。

---

## 4. 超参数组合建议

基于实验数据，推荐以下组合：

```text
推荐超参数组合

组合 A (保守稳定):
  entropy_coeff = 0.005
  kl_loss_coef  = 0.001
  ACTOR_LR      = 1e-6

组合 B (更强约束):
  entropy_coeff = 0.005
  kl_loss_coef  = 0.01
  ACTOR_LR      = 1e-6

调参顺序建议:
1. 先用默认参数跑 50 步，观察 entropy 趋势
2. 如果 entropy 下降过快 -> 加 entropy_coeff
3. 如果 entropy 失控飙升 -> 降 entropy_coeff
4. 如果策略漂移过大 -> 加 kl_loss_coef
```

---

## 课后练习

1. （判断题）entropy_coeff=0（默认值）时，模型没有任何探索保护机制，可能导致策略崩塌。

2. （判断题）entropy_coeff=0.01 比 entropy_coeff=0.005 更安全，因为更强的探索保证。

3. （判断题）RL 的学习率通常比 SFT 低 1-2 个数量级。

4. （单选题）entropy_coeff=0.01 导致 entropy 飙升到 4.51，说明什么？
    A. KL 正则太强
    B. Entropy bonus 过强，压过了策略梯度
    C. 学习率太低
    D. 训练正常

5. （单选题）调参时应该先观察什么？
    A. 直接设置最大 entropy_coeff
    B. 先用默认参数跑 50 步，观察 entropy 趋势
    C. 先调学习率
    D. 先调 KL coef

6. （多选题）以下哪些超参数影响 RL 训练稳定性？
    A. entropy_coeff
    B. kl_loss_coef
    C. ACTOR_LR
    D. ROLLOUT_N

In [ ]:
!cat ./answer/04.02_answer.txt